# FinRL-DeepSeek Backtest — v3 (Complete)

**vs v0 (original):** all fixes applied, nothing dropped  
**vs v2:** added back Llama agents, PPO/CPPO split results, actions+portfolio tracking, all 4 date vars

| | v0 | v2 | v3 |
|---|---|---|---|
| `fillna(3)` neutral | ❌ (used 0) | ✅ | ✅ |
| Signal normalisation cols | ❌ | ✅ | ✅ |
| `while not done` loop | ❌ | ✅ | ✅ |
| Actions + portfolio tracking | ✅ | ❌ | ✅ |
| Llama agents | ✅ | ❌ | ✅ |
| PPO / CPPO split result tables | ✅ | ❌ | ✅ |
| Sharpe / Sortino / Calmar / MaxDD | ❌ | ✅ | ✅ |
| Hyperparameter config + sweep | ❌ | ✅ | ✅ |
| 4 proper plots | ❌ | ✅ | ✅ |
| All 4 date vars (train + trade) | ✅ | ❌ | ✅ |


## Part 1 · Install & Setup

In [10]:
%pip install yfinance

Note: you may need to restart the kernel to use updated packages.


In [11]:
!pip install -q git+https://github.com/benstaf/FinRL.git
#!pip install -q selenium webdriver-manager alpaca-py datasets huggingface_hub yfinance




[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
import os, sys

if not os.path.exists('FinRL_LLM'):
    !git clone https://github.com/benstaf/FinRL_DeepSeek FinRL_LLM

os.chdir('FinRL_LLM')
sys.path.insert(0, os.getcwd())
print("CWD:", os.getcwd())
!ls


CWD: c:\Users\Mega-PC\Desktop\fin_AI_Project\FinRL_DeepSeek\FinRL_LLM


Cloning into 'FinRL_LLM'...
'ls' n'est pas reconnu en tant que commande interne
ou externe, un programme ex�cutable ou un fichier de commandes.


In [2]:
from huggingface_hub import snapshot_download
snapshot_download(
    repo_id="benstaf/Trading_agents",
    local_dir="trained_models",
    ignore_patterns=["*.md", "*.txt"]
)
!ls trained_models/


c:\Users\Mega-PC\Desktop\fin_AI_Project\.venv311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 16 files: 100%|██████████| 16/16 [00:00<00:00, 246.14it/s]
'ls' n'est pas reconnu en tant que commande interne
ou externe, un programme ex�cutable ou un fichier de commandes.


## Part 2 · Imports & Architecture

In [1]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.signal
import torch
import torch.nn as nn
from torch.distributions.normal import Normal
from torch.distributions.categorical import Categorical
from gymnasium.spaces import Box, Discrete
from datasets import load_dataset
from finrl.config import INDICATORS, TRAINED_MODEL_DIR
import yfinance as yf

from env_stocktrading import StockTradingEnv
from env_stocktrading_llm import StockTradingEnv as StockTradingEnv_llm
from env_stocktrading_llm_1 import StockTradingEnv as StockTradingEnv_llm_1
from env_stocktrading_llm_01 import StockTradingEnv as StockTradingEnv_llm_01
from env_stocktrading_llm_risk import StockTradingEnv as StockTradingEnv_llm_risk
from env_stocktrading_llm_risk_1 import StockTradingEnv as StockTradingEnv_llm_risk_1
from env_stocktrading_llm_risk_01 import StockTradingEnv as StockTradingEnv_llm_risk_01
from env_stocktrading_llm_risk_phase1 import StockTradingEnv as StockTradingEnv_llm_risk_phase1

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


Device: cuda


In [2]:
# Actor-Critic — must stay identical to training code so weights load correctly
def mlp(sizes, activation, output_activation=nn.Identity):
    layers = []
    for j in range(len(sizes) - 1):
        act = activation if j < len(sizes) - 2 else output_activation
        layers += [nn.Linear(sizes[j], sizes[j+1]), act()]
    return nn.Sequential(*layers)

class MLPGaussianActor(nn.Module):
    def __init__(self, obs_dim, act_dim, hidden_sizes, activation):
        super().__init__()
        self.log_std = nn.Parameter(torch.as_tensor(-0.5 * np.ones(act_dim, dtype=np.float32)))
        self.mu_net  = mlp([obs_dim] + list(hidden_sizes) + [act_dim], activation)
    def _distribution(self, obs):
        return Normal(self.mu_net(obs), torch.exp(self.log_std))
    def _log_prob_from_distribution(self, pi, act):
        return pi.log_prob(act).sum(axis=-1)
    def forward(self, obs, act=None):
        pi = self._distribution(obs)
        return pi, (self._log_prob_from_distribution(pi, act) if act is not None else None)

class MLPCritic(nn.Module):
    def __init__(self, obs_dim, hidden_sizes, activation):
        super().__init__()
        self.v_net = mlp([obs_dim] + list(hidden_sizes) + [1], activation)
    def forward(self, obs):
        return torch.squeeze(self.v_net(obs), -1)

class MLPActorCritic(nn.Module):
    def __init__(self, observation_space, action_space, hidden_sizes=(64,64), activation=nn.Tanh):
        super().__init__()
        obs_dim = observation_space.shape[0]
        self.pi = MLPGaussianActor(obs_dim, action_space.shape[0], hidden_sizes, activation)
        self.v  = MLPCritic(obs_dim, hidden_sizes, activation)
    def step(self, obs):
        with torch.no_grad():
            pi   = self.pi._distribution(obs)
            a    = pi.sample()
            logp = self.pi._log_prob_from_distribution(pi, a)
            v    = self.v(obs)
        return a.cpu().numpy(), v.cpu().numpy(), logp.cpu().numpy()
    def act(self, obs):
        return self.step(obs)[0]
    def act_deterministic(self, obs):
        with torch.no_grad():
            return self.pi.mu_net(obs).cpu().numpy()


## Part 3 · Date Config

In [3]:
# All 4 dates — train range for reference, trade range for backtest
TRAIN_START_DATE = '2013-01-01'
TRAIN_END_DATE   = '2018-12-31'
TRADE_START_DATE = '2019-01-01'
TRADE_END_DATE   = '2023-12-31'


## Part 4 · Load & Clean Trade Data

**Fixes vs v0:**
- `llm_sentiment`: `fillna(3)` not `fillna(0)` — 0 is outside the 1–5 scale, biases agent bearish
- `llm_risk`: already `fillna(3)` in v0, kept the same
- Added normalised signal columns for reference (`sent_z`, `risk_z`) — env files use raw columns


In [4]:
def load_trade_data(hf_file: str) -> pd.DataFrame:
    ds  = load_dataset("benstaf/nasdaq_2013_2023", data_files=hf_file)
    df  = pd.DataFrame(ds['train'])
    df  = df.drop(columns=[c for c in df.columns if 'Unnamed' in c], errors='ignore')
    uid = df['date'].unique()
    df['new_idx'] = df['date'].map({d: i for i, d in enumerate(uid)})
    return df.set_index('new_idx')

trade_baseline  = load_trade_data('trade_data_2019_2023.csv')
trade_sentiment = load_trade_data('trade_data_deepseek_sentiment_2019_2023.csv')
trade_risk      = load_trade_data('trade_data_deepseek_risk_2019_2023.csv')

# ── FIX: neutral fill for both LLM signals (scale is 1–5, neutral = 3) ───────
if 'llm_sentiment' in trade_sentiment.columns:
    trade_sentiment['llm_sentiment'] = trade_sentiment['llm_sentiment'].fillna(3)
if 'llm_sentiment' in trade_risk.columns:
    trade_risk['llm_sentiment'] = trade_risk['llm_sentiment'].fillna(3)
if 'llm_risk' in trade_risk.columns:
    trade_risk['llm_risk'] = trade_risk['llm_risk'].fillna(3)

# Normalised columns for inspection (env files use raw unless you retrain)
if 'llm_sentiment' in trade_sentiment.columns:
    trade_sentiment['sent_z'] = (trade_sentiment['llm_sentiment'] - 3) / 2   # → [-1, +1]
if 'llm_sentiment' in trade_risk.columns:
    trade_risk['sent_z'] = (trade_risk['llm_sentiment'] - 3) / 2
if 'llm_risk' in trade_risk.columns:
    trade_risk['risk_z'] = (trade_risk['llm_risk'] - 1) / 4                   # → [0, 1]

# Aliases matching v0 variable names (for backward compatibility)
trade_llm      = trade_sentiment
trade_llm_risk = trade_risk

print(f"Baseline  : {len(trade_baseline):,} rows")
print(f"Sentiment : {len(trade_sentiment):,} rows")
print(f"Risk      : {len(trade_risk):,} rows")

if 'llm_sentiment' in trade_sentiment.columns:
    print(f"\nSentiment value counts (after fillna):")
    print(trade_sentiment['llm_sentiment'].value_counts().sort_index())


Repo card metadata block was not found. Setting CardData to empty.
Repo card metadata block was not found. Setting CardData to empty.
Repo card metadata block was not found. Setting CardData to empty.


Baseline  : 105,588 rows
Sentiment : 105,588 rows
Risk      : 105,588 rows

Sentiment value counts (after fillna):
llm_sentiment
0.0       75
1.0      509
2.0     5131
3.0    86585
4.0    12321
5.0      967
Name: count, dtype: int64


## Part 5 · Hyperparameter Configuration

Edit these and re-run from Part 6 onwards.

| Param | Default | Notes |
|---|---|---|
| `TURBULENCE_THRESHOLD` | 70 | Lower = risk-off triggers sooner. Try 50/70/100 |
| `HMAX` | 100 | Max shares per trade. Try 20/50/100 |
| `TRANSACTION_COST` | 0.001 | 0.1% light, 0.002 more realistic |
| `REWARD_SCALING` | 1e-4 | Don't change for pre-trained weights |


In [5]:
TURBULENCE_THRESHOLD = 70
HMAX                 = 100
TRANSACTION_COST     = 0.001
REWARD_SCALING       = 1e-4

print(f"turbulence_threshold : {TURBULENCE_THRESHOLD}")
print(f"hmax                 : {HMAX}")
print(f"transaction_cost     : {TRANSACTION_COST*100:.2f}%")
print(f"reward_scaling       : {REWARD_SCALING}")


turbulence_threshold : 70
hmax                 : 100
transaction_cost     : 0.10%
reward_scaling       : 0.0001


## Part 6 · Build Trading Environments

In [6]:
def make_env_kwargs(trade_df, extra_llm_features=0):
    stock_dim   = len(trade_df.tic.unique())
    state_space = 1 + 2 * stock_dim + (len(INDICATORS) + extra_llm_features) * stock_dim
    return stock_dim, {
        "hmax"                : HMAX,
        "initial_amount"      : 1_000_000,
        "num_stock_shares"    : [0] * stock_dim,
        "buy_cost_pct"        : [TRANSACTION_COST] * stock_dim,
        "sell_cost_pct"       : [TRANSACTION_COST] * stock_dim,
        "state_space"         : state_space,
        "stock_dim"           : stock_dim,
        "tech_indicator_list" : INDICATORS,
        "action_space"        : stock_dim,
        "reward_scaling"      : REWARD_SCALING,
    }

_, env_kwargs_base = make_env_kwargs(trade_baseline,  extra_llm_features=0)
_, env_kwargs_llm  = make_env_kwargs(trade_sentiment, extra_llm_features=1)
_, env_kwargs_risk = make_env_kwargs(trade_risk,      extra_llm_features=2)

T = TURBULENCE_THRESHOLD

# Baseline
e_base = StockTradingEnv(df=trade_baseline, turbulence_threshold=T, risk_indicator_col='vix', **env_kwargs_base)

# PPO-DeepSeek (sentiment) — 10%, 1%, 0.1% modulation
e_llm_10  = StockTradingEnv_llm(df=trade_sentiment,    turbulence_threshold=T, risk_indicator_col='vix', **env_kwargs_llm)
e_llm_1   = StockTradingEnv_llm_1(df=trade_sentiment,  turbulence_threshold=T, risk_indicator_col='vix', **env_kwargs_llm)
e_llm_01  = StockTradingEnv_llm_01(df=trade_sentiment, turbulence_threshold=T, risk_indicator_col='vix', **env_kwargs_llm)

# CPPO-DeepSeek (risk) — 10%, 1%, 0.1% modulation
e_risk_10  = StockTradingEnv_llm_risk(df=trade_risk,    turbulence_threshold=T, risk_indicator_col='vix', **env_kwargs_risk)
e_risk_1   = StockTradingEnv_llm_risk_1(df=trade_risk,  turbulence_threshold=T, risk_indicator_col='vix', **env_kwargs_risk)
e_risk_01  = StockTradingEnv_llm_risk_01(df=trade_risk, turbulence_threshold=T, risk_indicator_col='vix', **env_kwargs_risk)
e_risk_phase1 = StockTradingEnv_llm_risk_phase1(
    df=trade_risk,
    turbulence_threshold=T,
    risk_indicator_col='vix',
    **env_kwargs_risk
)
# Llama agents reuse the same env types as their DeepSeek counterparts
e_llm_llama      = StockTradingEnv_llm(df=trade_sentiment, turbulence_threshold=T, risk_indicator_col='vix', **env_kwargs_llm)
e_risk_llama     = StockTradingEnv_llm_risk(df=trade_risk,  turbulence_threshold=T, risk_indicator_col='vix', **env_kwargs_risk)

observation_space      = e_base.observation_space
action_space           = e_base.action_space
observation_space_llm  = e_llm_10.observation_space
action_space_llm       = e_llm_10.action_space
observation_space_risk = e_risk_10.observation_space
action_space_risk      = e_risk_10.action_space
observation_space_risk_phase1 = e_risk_phase1.observation_space
action_space_risk_phase1      = e_risk_phase1.action_space

print("Environments built ✓")
print(f"  State shape (baseline) : {observation_space.shape}")
print(f"  State shape (LLM)      : {observation_space_llm.shape}")
print(f"  State shape (risk)     : {observation_space_risk.shape}")


Environments built ✓
  State shape (baseline) : (841,)
  State shape (LLM)      : (925,)
  State shape (risk)     : (1009,)


## Part 7 · Load Pre-trained Weights

**Full weight → agent mapping (all 10 agents):**

| File | Agent |
|---|---|
| `agent_ppo_100_epochs_20k_steps.pth` | PPO baseline |
| `agent_cppo_100_epochs_20k_steps.pth` | CPPO baseline |
| `agent_ppo_deepseek_100_epochs_20k_steps.pth` | PPO-DeepSeek 10% |
| `agent_ppo_deepseek_100_epochs_20k_steps_1.pth` | PPO-DeepSeek 1% |
| `agent_ppo_deepseek_100_epochs_20k_steps_01.pth` | PPO-DeepSeek 0.1% |
| `agent_cppo_deepseek_100_epochs_20k_steps.pth` | CPPO-DeepSeek 10% |
| `agent_cppo_deepseek_100_epochs_20k_steps_1.pth` | CPPO-DeepSeek 1% |
| `agent_cppo_deepseek_100_epochs_20k_steps_01.pth` | CPPO-DeepSeek 0.1% |
| `agent_ppo_llama_100_epochs_20k_steps.pth` | PPO-Llama |
| `agent_deepseek_20_epochs_20k_steps.pth` | CPPO-Llama (risk) |


In [7]:
def load_model(obs_space, act_space, path, hidden=(512, 512)):
    m = MLPActorCritic(obs_space, act_space, hidden_sizes=hidden)

    try:
        obj = torch.load(path, map_location=DEVICE)

        # Case 1: normal state_dict (.pth)
        if isinstance(obj, dict) and all(isinstance(k, str) for k in obj.keys()):
            m.load_state_dict(obj, strict=True)

        # Case 2: whole saved module (.pt from SpinUp pyt_save)
        elif isinstance(obj, nn.Module):
            m.load_state_dict(obj.state_dict(), strict=True)

        else:
            raise TypeError(f"Unsupported checkpoint content in {path}: {type(obj)}")

        m.to(DEVICE).eval()
        print(f"  ✓ {path}")
        return m

    except FileNotFoundError:
        print(f"  ✗ NOT FOUND: {path}")
        return None
    except Exception as e:
        print(f"  ✗ FAILED TO LOAD {path}: {e}")
        return None


MD = 'trained_models'

ppo_base     = load_model(observation_space,      action_space,      f'{MD}/agent_ppo_100_epochs_20k_steps.pth')
cppo_base    = load_model(observation_space,      action_space,      f'{MD}/agent_cppo_100_epochs_20k_steps.pth')
ppo_llm_10   = load_model(observation_space_llm,  action_space_llm,  f'{MD}/agent_ppo_deepseek_100_epochs_20k_steps.pth')
ppo_llm_1    = load_model(observation_space_llm,  action_space_llm,  f'{MD}/agent_ppo_deepseek_100_epochs_20k_steps_1.pth')
ppo_llm_01   = load_model(observation_space_llm,  action_space_llm,  f'{MD}/agent_ppo_deepseek_100_epochs_20k_steps_01.pth')
cppo_llm_10  = load_model(observation_space_risk, action_space_risk, f'{MD}/agent_cppo_deepseek_100_epochs_20k_steps.pth')
cppo_llm_1   = load_model(observation_space_risk, action_space_risk, f'{MD}/agent_cppo_deepseek_100_epochs_20k_steps_1.pth')
cppo_llm_01  = load_model(observation_space_risk, action_space_risk, f'{MD}/agent_cppo_deepseek_100_epochs_20k_steps_01.pth')
ppo_llama    = load_model(observation_space_llm,  action_space_llm,  f'{MD}/agent_ppo_llama_100_epochs_20k_steps.pth')
cppo_llama   = load_model(observation_space_risk, action_space_risk, f'{MD}/agent_deepseek_20_epochs_20k_steps.pth')

# YOUR PHASE 1 MODEL
cppo_phase1_ep90 = load_model(
    observation_space_risk_phase1,
    action_space_risk_phase1,
    'results/cppo_phase1_continue_ep90/pyt_save/model.pt',
    hidden=(512, 512)
)
cppo_phase1_ep100 = load_model(
    observation_space_risk_phase1,
    action_space_risk_phase1,
    'results/cppo_phase1_continue_ep100/pyt_save/model.pt',
    hidden=(512, 512)
)
cppo_phase1_ep110 = load_model(
    observation_space_risk_phase1,
    action_space_risk_phase1,
    'results/cppo_phase1_continue_s0/pyt_save/model.pt',
    hidden=(512, 512)
)

  ✓ trained_models/agent_ppo_100_epochs_20k_steps.pth
  ✓ trained_models/agent_cppo_100_epochs_20k_steps.pth
  ✓ trained_models/agent_ppo_deepseek_100_epochs_20k_steps.pth
  ✓ trained_models/agent_ppo_deepseek_100_epochs_20k_steps_1.pth
  ✓ trained_models/agent_ppo_deepseek_100_epochs_20k_steps_01.pth
  ✓ trained_models/agent_cppo_deepseek_100_epochs_20k_steps.pth
  ✓ trained_models/agent_cppo_deepseek_100_epochs_20k_steps_1.pth
  ✓ trained_models/agent_cppo_deepseek_100_epochs_20k_steps_01.pth
  ✓ trained_models/agent_ppo_llama_100_epochs_20k_steps.pth
  ✓ trained_models/agent_deepseek_20_epochs_20k_steps.pth
  ✓ results/cppo_phase1_continue_ep90/pyt_save/model.pt
  ✓ results/cppo_phase1_continue_ep100/pyt_save/model.pt
  ✓ results/cppo_phase1_continue_s0/pyt_save/model.pt


## Part 8 · Run Backtest

Returns **4 outputs per agent** (matching v0 structure):
- `assets` — total portfolio value per day
- `account_memory` — same as assets (daily)
- `actions_memory` — what the agent traded each day
- `portfolio_distribution` — cash + per-stock fractions


In [8]:
def drl_predict(model, env):
    """
    Run one full backtest episode.
    Returns: (episode_total_assets, account_memory, actions_memory, portfolio_distribution)
    """
    state, _ = env.reset()
    account_memory       = []
    actions_memory       = []
    portfolio_dist       = []
    episode_total_assets = [env.initial_amount]
    done = False

    with torch.no_grad():
        while not done:
            s_t = torch.tensor(state, dtype=torch.float32, device=DEVICE).unsqueeze(0)
            #pi, _ = model.pi(s_t)
            #action = pi.sample().cpu().numpy().flatten()
            action = model.pi.mu_net(s_t).cpu().numpy().flatten()

            state, _, terminated, truncated, _ = env.step(action)
            done = terminated or truncated

            prices       = env.df.loc[env.day, 'close'].values
            cash         = env.asset_memory[-1]
            holdings     = env.num_stock_shares
            total_asset  = cash + (prices * holdings).sum()

            stock_values = prices * holdings
            distribution = {
                "cash"   : cash / total_asset,
                "stocks" : (stock_values / total_asset).tolist()
            }

            episode_total_assets.append(total_asset)
            account_memory.append(total_asset)
            actions_memory.append(action)
            portfolio_dist.append(distribution)

    return episode_total_assets, account_memory, actions_memory, portfolio_dist


# Run all agents (skip if model failed to load)
print("Running backtests...")

agent_map = {
    'PPO'                : (ppo_base,    e_base),
    'CPPO'               : (cppo_base,   e_base),
    'PPO-DeepSeek 10%'   : (ppo_llm_10,  e_llm_10),
    'PPO-DeepSeek 1%'    : (ppo_llm_1,   e_llm_1),
    'PPO-DeepSeek 0.1%'  : (ppo_llm_01,  e_llm_01),
    'CPPO-DeepSeek 10%'  : (cppo_llm_10, e_risk_10),
    'CPPO-DeepSeek 1%'   : (cppo_llm_1,  e_risk_1),
    'CPPO-DeepSeek 0.1%' : (cppo_llm_01, e_risk_01),
    #'PPO-Llama'          : (ppo_llama,   e_llm_llama),
    #'CPPO-Llama'         : (cppo_llama,  e_risk_llama),
    'CPPO-DeepSeek Phase1 EP90' : (cppo_phase1_ep90, e_risk_phase1),
    'CPPO-DeepSeek Phase1 EP100' : (cppo_phase1_ep100, e_risk_phase1),
    'CPPO-DeepSeek Phase1 EP110' : (cppo_phase1_ep110, e_risk_phase1),

}

results_raw = {}
for name, (model, env) in agent_map.items():
    if model is None:
        print(f"  ✗ {name} skipped (model not loaded)")
        continue
    assets, acct, actions, dist = drl_predict(model, env)
    results_raw[name] = {
        'assets'    : assets,
        'account'   : acct,
        'actions'   : actions,
        'portfolio' : dist,
    }
    ret = (assets[-1] / assets[0] - 1) * 100
    print(f"  ✓ {name:<25} final: ${assets[-1]/1e6:.3f}M  ({ret:+.1f}%)")


Running backtests...
  ✓ PPO                       final: $3.813M  (+281.3%)
  ✓ CPPO                      final: $2.394M  (+139.4%)
  ✓ PPO-DeepSeek 10%          final: $1.821M  (+82.1%)
  ✓ PPO-DeepSeek 1%           final: $1.091M  (+9.1%)
  ✓ PPO-DeepSeek 0.1%         final: $2.212M  (+121.2%)
  ✓ CPPO-DeepSeek 10%         final: $1.434M  (+43.4%)
  ✓ CPPO-DeepSeek 1%          final: $2.608M  (+160.8%)
  ✓ CPPO-DeepSeek 0.1%        final: $0.699M  (-30.1%)
  ✓ CPPO-DeepSeek Phase1 EP90 final: $2.182M  (+118.2%)
  ✓ CPPO-DeepSeek Phase1 EP100 final: $2.184M  (+118.4%)
  ✓ CPPO-DeepSeek Phase1 EP110 final: $1.898M  (+89.8%)


In [25]:
selected_agent_map = {
    "PPO": (ppo_base, e_base),
    "CPPO": (cppo_base, e_base),
    "PPO-DeepSeek 0.1%": (ppo_llm_01, e_llm_01),
    "CPPO-DeepSeek 1%": (cppo_llm_1, e_risk_1),

    "CPPO-DeepSeek Phase1 EP90": (cppo_phase1_ep90, e_risk_phase1),
    "CPPO-DeepSeek Phase1 EP100": (cppo_phase1_ep100, e_risk_phase1),
    "CPPO-DeepSeek Phase1 EP110": (cppo_phase1_ep110, e_risk_phase1),
}

def drl_predict_stochastic(model, env, seed=0):
    """
    Run one stochastic backtest episode using pi.sample().
    """
    torch.manual_seed(seed)
    np.random.seed(seed)

    model.eval()

    state, _ = env.reset()
    assets = [env.initial_amount]
    done = False

    with torch.no_grad():
        while not done:
            s_t = torch.tensor(state, dtype=torch.float32, device=DEVICE).unsqueeze(0)

            pi, _ = model.pi(s_t)

            # Stochastic policy evaluation
            action = pi.sample().cpu().numpy().flatten()

            # Respect action space
            action = np.clip(action, -1, 1)

            state, _, terminated, truncated, _ = env.step(action)
            done = terminated or truncated

            assets.append(env.asset_memory[-1])

    return np.array(assets)

def compute_backtest_metrics(assets, trading_days=252):
    assets = np.asarray(assets, dtype=np.float64)
    returns = pd.Series(assets).pct_change().dropna()

    total_return = (assets[-1] / assets[0] - 1) * 100

    n_days = len(assets) - 1
    ann_return = ((assets[-1] / assets[0]) ** (trading_days / n_days) - 1) * 100

    sharpe = np.nan
    sortino = np.nan
    calmar = np.nan

    if returns.std() != 0:
        sharpe = np.sqrt(trading_days) * returns.mean() / returns.std()

    downside = returns[returns < 0]
    if len(downside) > 0 and downside.std() != 0:
        sortino = np.sqrt(trading_days) * returns.mean() / downside.std()

    running_max = np.maximum.accumulate(assets)
    drawdowns = assets / running_max - 1
    max_dd = drawdowns.min() * 100

    if max_dd != 0:
        calmar = ann_return / abs(max_dd)

    win_rate = (returns > 0).mean() * 100

    cvar_5 = returns[returns <= returns.quantile(0.05)].mean() * 100

    return {
        "Ann Return %": ann_return,
        "Total Return %": total_return,
        "Sharpe": sharpe,
        "Sortino": sortino,
        "Calmar": calmar,
        "Max DD %": max_dd,
        "Win Rate %": win_rate,
        "CVaR 5%": cvar_5,
    }

N_RUNS = 10   # change to 30 if you want

all_stochastic_rows = []
all_stochastic_assets = {}

for name, (model, env) in selected_agent_map.items():
    if model is None:
        print(f"✗ {name} skipped because model is None")
        continue

    print(f"\nRunning stochastic tests for: {name}")

    run_metrics = []
    run_assets = []

    for seed in range(N_RUNS):
        assets = drl_predict_stochastic(model, env, seed=seed)
        metrics = compute_backtest_metrics(assets)

        run_assets.append(assets)
        run_metrics.append(metrics)

        print(
            f"  seed {seed:02d} | "
            f"final=${assets[-1]/1e6:.3f}M | "
            f"total_return={metrics['Total Return %']:.2f}% | "
            f"sharpe={metrics['Sharpe']:.3f}"
        )

    all_stochastic_assets[name] = run_assets

    df_m = pd.DataFrame(run_metrics)

    row = {"Strategy": name}

    for col in df_m.columns:
        row[f"{col} Mean"] = df_m[col].mean()
        row[f"{col} Std"] = df_m[col].std()
        row[f"{col} Min"] = df_m[col].min()
        row[f"{col} Max"] = df_m[col].max()

    all_stochastic_rows.append(row)

stochastic_summary = pd.DataFrame(all_stochastic_rows)
stochastic_summary

main_cols = [
    "Strategy",
    "Ann Return % Mean", "Ann Return % Std",
    "Total Return % Mean", "Total Return % Std",
    "Sharpe Mean", "Sharpe Std",
    "Sortino Mean", "Sortino Std",
    "Calmar Mean", "Calmar Std",
    "Max DD % Mean", "Max DD % Std",
    "Win Rate % Mean", "Win Rate % Std",
    "CVaR 5% Mean", "CVaR 5% Std",
]

stochastic_summary_main = stochastic_summary[main_cols].copy()

# Optional rounding
for col in stochastic_summary_main.columns:
    if col != "Strategy":
        stochastic_summary_main[col] = stochastic_summary_main[col].round(3)

stochastic_summary_main


Running stochastic tests for: PPO
day: 1256, episode: 10
begin_total_asset: 1000000.00
end_total_asset: 2750885.61
total_reward: 1750885.61
total_cost: 43684.69
total_trades: 56611
Sharpe: 1.014
  seed 00 | final=$2.751M | total_return=175.09% | sharpe=1.014
  seed 01 | final=$3.176M | total_return=217.56% | sharpe=1.060
  seed 02 | final=$3.543M | total_return=254.33% | sharpe=1.131
  seed 03 | final=$4.043M | total_return=304.28% | sharpe=1.203
  seed 04 | final=$3.373M | total_return=237.31% | sharpe=1.113
  seed 05 | final=$3.205M | total_return=220.52% | sharpe=1.027
  seed 06 | final=$3.522M | total_return=252.16% | sharpe=1.141
  seed 07 | final=$3.125M | total_return=212.46% | sharpe=0.915
  seed 08 | final=$3.679M | total_return=267.92% | sharpe=1.250
  seed 09 | final=$3.817M | total_return=281.74% | sharpe=1.237

Running stochastic tests for: CPPO
day: 1256, episode: 20
begin_total_asset: 1000000.00
end_total_asset: 1677793.72
total_reward: 677793.72
total_cost: 17759.30
to

,Strategy,Ann Return % Mean,Ann Return % Std,Total Return % Mean,Total Return % Std,Sharpe Mean,Sharpe Std,Sortino Mean,Sortino Std,Calmar Mean,Calmar Std,Max DD % Mean,Max DD % Std,Win Rate % Mean,Win Rate % Std,CVaR 5% Mean,CVaR 5% Std
0,PPO,27.867,2.860,242.336,37.733,1.109,0.107,1.528,0.149,0.991,0.183,-28.812,5.198,55.036,0.424,-3.566,0.298
1,CPPO,13.366,1.379,87.213,11.202,0.650,0.047,0.902,0.064,0.435,0.045,-30.803,2.547,51.488,0.556,-3.471,0.092
2,PPO-DeepSeek 0.1%,22.378,2.349,174.734,26.703,0.880,0.086,1.296,0.133,0.595,0.102,-38.100,3.839,53.484,0.392,-3.841,0.179
3,CPPO-DeepSeek 1%,20.208,3.378,152.219,35.439,0.910,0.126,1.256,0.186,0.791,0.187,-25.989,2.314,54.288,0.531,-3.438,0.093
4,CPPO-DeepSeek Phase1 EP90,16.843,3.517,119.164,33.825,0.878,0.173,1.196,0.193,0.653,0.157,-25.921,1.671,54.208,1.245,-2.912,0.122
5,CPPO-DeepSeek Phase1 EP100,16.094,1.848,110.992,16.880,0.772,0.090,1.079,0.117,0.586,0.111,-27.990,3.672,53.731,0.613,-3.287,0.204
6,CPPO-DeepSeek Phase1 EP110,16.865,3.216,119.023,28.720,0.760,0.115,1.018,0.151,0.531,0.128,-32.404,4.545,54.336,0.853,-3.641,0.235


In [2]:
stochastic_summary_main = pd.DataFrame([
    ["PPO", 27.867, 2.860, 242.336, 37.733, 1.109, 0.107, 1.528, 0.149, 0.991, 0.183, -28.812, 5.198, 55.036, 0.424, -3.566, 0.298],
    ["CPPO", 13.366, 1.379, 87.213, 11.202, 0.650, 0.047, 0.902, 0.064, 0.435, 0.045, -30.803, 2.547, 51.488, 0.556, -3.471, 0.092],
    ["PPO-DeepSeek 0.1%", 22.378, 2.349, 174.734, 26.703, 0.880, 0.086, 1.296, 0.133, 0.595, 0.102, -38.100, 3.839, 53.484, 0.392, -3.841, 0.179],
    ["CPPO-DeepSeek 1%", 20.208, 3.378, 152.219, 35.439, 0.910, 0.126, 1.256, 0.186, 0.791, 0.187, -25.989, 2.314, 54.288, 0.531, -3.438, 0.093],
    ["CPPO-DeepSeek Phase1 EP90", 16.843, 3.517, 119.164, 33.825, 0.878, 0.173, 1.196, 0.193, 0.653, 0.157, -25.921, 1.671, 54.208, 1.245, -2.912, 0.122],
    ["CPPO-DeepSeek Phase1 EP100", 16.094, 1.848, 110.992, 16.880, 0.772, 0.090, 1.079, 0.117, 0.586, 0.111, -27.990, 3.672, 53.731, 0.613, -3.287, 0.204],
    ["CPPO-DeepSeek Phase1 EP110", 16.865, 3.216, 119.023, 28.720, 0.760, 0.115, 1.018, 0.151, 0.531, 0.128, -32.404, 4.545, 54.336, 0.853, -3.641, 0.235],
], columns=[
    "Strategy",
    "Ann Return % Mean", "Ann Return % Std",
    "Total Return % Mean", "Total Return % Std",
    "Sharpe Mean", "Sharpe Std",
    "Sortino Mean", "Sortino Std",
    "Calmar Mean", "Calmar Std",
    "Max DD % Mean", "Max DD % Std",
    "Win Rate % Mean", "Win Rate % Std",
    "CVaR 5% Mean", "CVaR 5% Std",
])

Path("final_report_exports").mkdir(exist_ok=True)
stochastic_summary_main.to_csv("final_report_exports/stochastic_summary_main_reconstructed.csv", index=False)

stochastic_summary_main

,Strategy,Ann Return % Mean,Ann Return % Std,Total Return % Mean,Total Return % Std,Sharpe Mean,Sharpe Std,Sortino Mean,Sortino Std,Calmar Mean,Calmar Std,Max DD % Mean,Max DD % Std,Win Rate % Mean,Win Rate % Std,CVaR 5% Mean,CVaR 5% Std
0,PPO,27.867,2.860,242.336,37.733,1.109,0.107,1.528,0.149,0.991,0.183,-28.812,5.198,55.036,0.424,-3.566,0.298
1,CPPO,13.366,1.379,87.213,11.202,0.650,0.047,0.902,0.064,0.435,0.045,-30.803,2.547,51.488,0.556,-3.471,0.092
2,PPO-DeepSeek 0.1%,22.378,2.349,174.734,26.703,0.880,0.086,1.296,0.133,0.595,0.102,-38.100,3.839,53.484,0.392,-3.841,0.179
3,CPPO-DeepSeek 1%,20.208,3.378,152.219,35.439,0.910,0.126,1.256,0.186,0.791,0.187,-25.989,2.314,54.288,0.531,-3.438,0.093
4,CPPO-DeepSeek Phase1 EP90,16.843,3.517,119.164,33.825,0.878,0.173,1.196,0.193,0.653,0.157,-25.921,1.671,54.208,1.245,-2.912,0.122
5,CPPO-DeepSeek Phase1 EP100,16.094,1.848,110.992,16.880,0.772,0.090,1.079,0.117,0.586,0.111,-27.990,3.672,53.731,0.613,-3.287,0.204
6,CPPO-DeepSeek Phase1 EP110,16.865,3.216,119.023,28.720,0.760,0.115,1.018,0.151,0.531,0.128,-32.404,4.545,54.336,0.853,-3.641,0.235


✗ metrics not found in memory
✗ result not found in memory
✗ results_raw not found in memory
✗ stochastic_summary_main not found in memory
✗ stochastic_summary not found in memory
✗ all_stochastic_assets not found in memory
✗ all_stochastic_rows not found in memory

Saved files:


## Part 9 · NASDAQ-100 Benchmark

In [9]:
import yfinance as yf
import pandas as pd

df_dji = yf.download("^NDX", start=TRADE_START_DATE, end=TRADE_END_DATE, auto_adjust=False)

df_dji = df_dji.reset_index()[["Date", "Close"]]
df_dji.columns = ["date", "close"]
df_dji["date"] = pd.to_datetime(df_dji["date"]).dt.strftime("%Y-%m-%d")

fst_day = df_dji["close"].iloc[0]
df_dji["ndx_norm"] = df_dji["close"] / fst_day * 1_000_000

print(f"NDX: {len(df_dji)} trading days")
df_dji.tail(3)

[*********************100%***********************]  1 of 1 completed

NDX: 1258 trading days


,date,close,ndx_norm
1255,2023-12-27,16906.800781,2.657938e+06
1256,2023-12-28,16898.470703,2.656629e+06
1257,2023-12-29,16825.929688,2.645225e+06


## Part 10 · Build Result DataFrames

Three views (matching v0 structure + combined):
- `result` — all strategies together
- `result_ppo` — PPO family vs benchmark  
- `result_cppo` — CPPO family vs benchmark


In [10]:
def build_result(agent_names, results_raw, df_dji):
    """Build aligned result DataFrame for a list of agent names."""
    ndx = df_dji.set_index(pd.to_datetime(df_dji['date']))['ndx_norm']
    frames = {}
    for name in agent_names:
        if name not in results_raw:
            continue
        if   'CPPO' in name and ('DeepSeek' in name or 'Llama' in name):
            trade_ref = trade_risk
        elif 'DeepSeek' in name or 'Llama' in name:
            trade_ref = trade_sentiment
        else:
            trade_ref = trade_baseline
        dates = pd.to_datetime(trade_ref['date'].unique())
        vals  = results_raw[name]['assets']
        s     = pd.Series(vals[1:], index=dates)
        frames[name] = s / s.iloc[0] * 1_000_000

    df = pd.DataFrame(frames)
    df['Nasdaq-100'] = ndx.reindex(df.index)
    return df.dropna()

selected_agents = [
    "PPO",
    "CPPO",
    "PPO-DeepSeek 0.1%",
    "CPPO-DeepSeek 1%",
    "CPPO-DeepSeek Phase1 EP90",
    "CPPO-DeepSeek Phase1 EP100",
    "CPPO-DeepSeek Phase1 EP110",
]

available = list(results_raw.keys())

result = build_result(available, results_raw, df_dji)

ppo_agents  = [a for a in available if 'CPPO' not in a]
cppo_agents = [a for a in available if 'CPPO' in a]
best_agents = [a for a in selected_agents if a in available]

result_ppo  = build_result(ppo_agents,  results_raw, df_dji)
result_cppo = build_result(cppo_agents, results_raw, df_dji)
result_best = build_result(best_agents, results_raw, df_dji)

print(f"result      : {result.shape}")
print(f"result_ppo  : {result_ppo.shape}")
print(f"result_cppo : {result_cppo.shape}")
print(f"result_best : {result_best.shape}")
result.head(3)


result      : (1257, 12)
result_ppo  : (1257, 5)
result_cppo : (1257, 8)
result_best : (1257, 8)


,PPO,CPPO,PPO-DeepSeek 10%,PPO-DeepSeek 1%,PPO-DeepSeek 0.1%,CPPO-DeepSeek 10%,CPPO-DeepSeek 1%,CPPO-DeepSeek 0.1%,CPPO-DeepSeek Phase1 EP90,CPPO-DeepSeek Phase1 EP100,CPPO-DeepSeek Phase1 EP110,Nasdaq-100
2019-01-02,1.000000e+06,1.000000e+06,1.000000e+06,1.000000e+06,1.000000e+06,1.000000e+06,1.000000e+06,1.000000e+06,1.000000e+06,1.000000e+06,1.000000e+06,1.000000e+06
2019-01-03,1.025377e+06,1.029278e+06,1.029674e+06,1.028251e+06,1.031618e+06,1.040681e+06,1.035534e+06,1.022799e+06,1.032531e+06,1.037254e+06,1.036303e+06,9.663976e+05
2019-01-04,1.035760e+06,1.048270e+06,1.050518e+06,1.048743e+06,1.054061e+06,1.046596e+06,1.052462e+06,1.039107e+06,1.047735e+06,1.053779e+06,1.053364e+06,1.009716e+06


## Part 11 · Performance Metrics

In [12]:
def compute_metrics(df: pd.DataFrame, benchmark='Nasdaq-100', rf=0.0) -> pd.DataFrame:
    rows = []
    bench_ret = df[benchmark].pct_change().dropna()
    n_years   = len(bench_ret) / 252

    for col in df.columns:
        if col == benchmark:
            continue
        ret = df[col].pct_change().dropna()
        ret, b = ret.align(bench_ret, join='inner')

        total_ret = df[col].iloc[-1] / df[col].iloc[0] - 1
        ann_ret   = (1 + total_ret) ** (1 / n_years) - 1
        excess    = ret - rf / 252
        sharpe    = np.sqrt(252) * excess.mean() / excess.std()
        down      = ret[ret < 0]
        sortino   = np.sqrt(252) * excess.mean() / down.std() if len(down) > 1 else np.nan
        cum       = (1 + ret).cumprod()
        max_dd    = ((cum - cum.cummax()) / cum.cummax()).min()
        calmar    = ann_ret / abs(max_dd) if max_dd != 0 else np.nan
        win_rate  = (ret > 0).mean()
        exc_b     = ret - b
        ir        = np.sqrt(252) * exc_b.mean() / exc_b.std()
        var5      = np.percentile(ret, 5)
        cvar      = ret[ret <= var5].mean()
        up95      = np.percentile(ret, 95)
        rachev    = ret[ret >= up95].mean() / abs(cvar) if cvar != 0 else np.nan

        rows.append({
            'Strategy'       : col,
            'Ann Return %'   : round(ann_ret  * 100, 2),
            'Total Return %' : round(total_ret * 100, 2),
            'Sharpe'         : round(sharpe, 3),
            'Sortino'        : round(sortino, 3),
            'Calmar'         : round(calmar, 3),
            'Max DD %'       : round(max_dd   * 100, 2),
            'Win Rate %'     : round(win_rate  * 100, 2),
            'Info Ratio'     : round(ir, 3),
            'CVaR 5%'        : round(cvar * 100, 4),
            'Rachev'         : round(rachev, 3),
        })

    return pd.DataFrame(rows).set_index('Strategy')

metrics      = compute_metrics(result)
metrics_ppo  = compute_metrics(result_ppo)
metrics_cppo = compute_metrics(result_cppo)
best_metrics  = compute_metrics(result_best)

print("=== ALL STRATEGIES ===")
display(metrics.style
    .background_gradient(subset=['Ann Return %','Sharpe','Sortino','Calmar','Info Ratio'], cmap='RdYlGn')
    .background_gradient(subset=['Max DD %'], cmap='RdYlGn_r')
    .format({c: '{:.3f}' for c in metrics.select_dtypes('number').columns})
    .set_caption(f'All agents | turb={TURBULENCE_THRESHOLD} | hmax={HMAX} | cost={TRANSACTION_COST}'))


=== ALL STRATEGIES ===


,Ann Return %,Total Return %,Sharpe,Sortino,Calmar,Max DD %,Win Rate %,Info Ratio,CVaR 5%,Rachev
Strategy,,,,,,,,,,
PPO,30.980,283.810,1.010,1.387,0.787,-39.380,54.380,0.215,-4.703,0.988
CPPO,19.250,140.460,0.732,0.991,0.428,-45.010,54.300,-0.017,-4.490,0.973
PPO-DeepSeek 10%,12.830,82.490,0.612,0.874,0.396,-32.420,52.390,-0.213,-3.487,1.043
PPO-DeepSeek 1%,1.800,9.320,0.213,0.305,0.029,-63.260,51.110,-0.396,-4.458,1.011
PPO-DeepSeek 0.1%,17.340,121.860,0.790,1.092,0.563,-30.780,53.500,-0.112,-3.529,0.937
CPPO-DeepSeek 10%,8.010,46.790,0.416,0.540,0.196,-40.850,52.790,-0.288,-4.325,0.855
CPPO-DeepSeek 1%,21.480,163.760,0.993,1.390,0.803,-26.750,53.420,-0.030,-3.178,0.978
CPPO-DeepSeek 0.1%,-6.880,-29.890,-0.017,-0.023,-0.096,-71.680,51.110,-0.526,-5.294,0.975
CPPO-DeepSeek Phase1 EP90,17.120,119.860,0.750,1.028,0.533,-32.120,54.220,-0.104,-3.728,0.947


In [13]:
print("=== PPO FAMILY ===")
display(metrics_ppo.style
    .background_gradient(subset=['Ann Return %','Sharpe','Sortino','Calmar','Info Ratio'], cmap='RdYlGn')
    .background_gradient(subset=['Max DD %'], cmap='RdYlGn_r')
    .format({c: '{:.3f}' for c in metrics_ppo.select_dtypes('number').columns}))

print("\n=== CPPO FAMILY ===")
display(metrics_cppo.style
    .background_gradient(subset=['Ann Return %','Sharpe','Sortino','Calmar','Info Ratio'], cmap='RdYlGn')
    .background_gradient(subset=['Max DD %'], cmap='RdYlGn_r')
    .format({c: '{:.3f}' for c in metrics_cppo.select_dtypes('number').columns}))

print("\n=== SELECTED BEST AGENTS ===")
display(best_metrics.style
    .background_gradient(subset=['Ann Return %','Sharpe','Sortino','Calmar','Info Ratio'], cmap='RdYlGn')
    .background_gradient(subset=['Max DD %'], cmap='RdYlGn_r')
    .format({c: '{:.3f}' for c in best_metrics.select_dtypes('number').columns}))   


=== PPO FAMILY ===


,Ann Return %,Total Return %,Sharpe,Sortino,Calmar,Max DD %,Win Rate %,Info Ratio,CVaR 5%,Rachev
Strategy,,,,,,,,,,
PPO,30.980,283.810,1.010,1.387,0.787,-39.380,54.380,0.215,-4.703,0.988
PPO-DeepSeek 10%,12.830,82.490,0.612,0.874,0.396,-32.420,52.390,-0.213,-3.487,1.043
PPO-DeepSeek 1%,1.800,9.320,0.213,0.305,0.029,-63.260,51.110,-0.396,-4.458,1.011
PPO-DeepSeek 0.1%,17.340,121.860,0.790,1.092,0.563,-30.780,53.500,-0.112,-3.529,0.937



=== CPPO FAMILY ===


,Ann Return %,Total Return %,Sharpe,Sortino,Calmar,Max DD %,Win Rate %,Info Ratio,CVaR 5%,Rachev
Strategy,,,,,,,,,,
CPPO,19.250,140.460,0.732,0.991,0.428,-45.010,54.300,-0.017,-4.490,0.973
CPPO-DeepSeek 10%,8.010,46.790,0.416,0.540,0.196,-40.850,52.790,-0.288,-4.325,0.855
CPPO-DeepSeek 1%,21.480,163.760,0.993,1.390,0.803,-26.750,53.420,-0.030,-3.178,0.978
CPPO-DeepSeek 0.1%,-6.880,-29.890,-0.017,-0.023,-0.096,-71.680,51.110,-0.526,-5.294,0.975
CPPO-DeepSeek Phase1 EP90,17.120,119.860,0.750,1.028,0.533,-32.120,54.220,-0.104,-3.728,0.947
CPPO-DeepSeek Phase1 EP100,17.180,120.350,0.786,1.036,0.423,-40.590,54.460,-0.117,-3.588,0.900
CPPO-DeepSeek Phase1 EP110,13.910,91.390,0.619,0.842,0.303,-45.860,54.220,-0.164,-3.973,0.933



=== SELECTED BEST AGENTS ===


,Ann Return %,Total Return %,Sharpe,Sortino,Calmar,Max DD %,Win Rate %,Info Ratio,CVaR 5%,Rachev
Strategy,,,,,,,,,,
PPO,30.980,283.810,1.010,1.387,0.787,-39.380,54.380,0.215,-4.703,0.988
CPPO,19.250,140.460,0.732,0.991,0.428,-45.010,54.300,-0.017,-4.490,0.973
PPO-DeepSeek 0.1%,17.340,121.860,0.790,1.092,0.563,-30.780,53.500,-0.112,-3.529,0.937
CPPO-DeepSeek 1%,21.480,163.760,0.993,1.390,0.803,-26.750,53.420,-0.030,-3.178,0.978
CPPO-DeepSeek Phase1 EP90,17.120,119.860,0.750,1.028,0.533,-32.120,54.220,-0.104,-3.728,0.947
CPPO-DeepSeek Phase1 EP100,17.180,120.350,0.786,1.036,0.423,-40.590,54.460,-0.117,-3.588,0.900
CPPO-DeepSeek Phase1 EP110,13.910,91.390,0.619,0.842,0.303,-45.860,54.220,-0.164,-3.973,0.933


## Part 12 · Visualisations

In [18]:
plt.style.use('seaborn-v0_8-darkgrid')

def plot_portfolio(df, title_suffix=''):
    fig, ax = plt.subplots(figsize=(16, 6))
    for col in df.columns:
        ax.plot(df.index, df[col] / 1e6, label=col,
                lw=2.5 if col=='Nasdaq-100' else 1.5,
                ls='--' if col=='Nasdaq-100' else '-')
    ax.set_title(f'Portfolio Value {title_suffix}', fontsize=13, fontweight='bold')
    ax.set_xlabel('Date'); ax.set_ylabel('Value ($M)')
    ax.legend(fontsize=8, loc='upper left')
    plt.tight_layout()
    fname = f"plot_portfolio_{title_suffix.replace(' ','_').replace('|','').strip()}.png"
    plt.savefig(fname, dpi=150); plt.show()

plot_portfolio(result,      f'— All | turb={TURBULENCE_THRESHOLD} hmax={HMAX} cost={TRANSACTION_COST}')
plot_portfolio(result_ppo,  '— PPO Family')
plot_portfolio(result_cppo, '— CPPO Family')
plot_portfolio(result_best, '— Main Comparison Strategies')


In [22]:
# Metric bar charts
def plot_metric_bars(metrics_df, title=''):
    metric_cols = ['Ann Return %', 'Sharpe', 'Sortino', 'Calmar', 'Max DD %', 'CVaR 5%']
    titles      = ['Ann. Return (%)', 'Sharpe', 'Sortino', 'Calmar', 'Max Drawdown (%)', 'CVaR 5%']
    fig, axes = plt.subplots(2, 3, figsize=(18, 9))
    for ax, col, t in zip(axes.flatten(), metric_cols, titles):
        asc  = (col == 'Max DD %')
        vals = metrics_df[col].sort_values(ascending=asc)
        colors = ['#d62728' if v < 0 else '#2196F3' for v in vals]
        bars = ax.barh(vals.index, vals.values, color=colors, edgecolor='white', height=0.6)
        ax.axvline(0, color='black', lw=0.8)
        ax.set_title(t, fontsize=11, fontweight='bold')
        for bar, v in zip(bars, vals.values):
            ax.text(bar.get_width() + 0.01*(vals.abs().max() or 1),
                    bar.get_y()+bar.get_height()/2, f'{v:.2f}', va='center', fontsize=7)
        ax.tick_params(axis='y', labelsize=7)
    plt.suptitle(f'Performance Metrics {title}', fontsize=13, fontweight='bold')
    plt.tight_layout(); plt.savefig(f'plot_metrics_{title}.png', dpi=150); plt.show()

plot_metric_bars(metrics,      'all')
plot_metric_bars(metrics_ppo,  'ppo')
plot_metric_bars(metrics_cppo, 'cppo')
plot_metric_bars(best_metrics, 'Main Comparison Strategies')


In [20]:
# Rolling 60-day Sharpe
fig, ax = plt.subplots(figsize=(16, 5))
ret_df = result.pct_change().dropna()
for col in ret_df.columns:
    rs = ret_df[col].rolling(60).apply(lambda x: np.sqrt(252)*x.mean()/x.std() if x.std()>0 else 0)
    ax.plot(rs.index, rs, label=col,
            lw=2.5 if col=='Nasdaq-100' else 1.2,
            ls='--' if col=='Nasdaq-100' else '-')
ax.axhline(0, color='black', lw=0.8, ls=':')
ax.set_title('Rolling 60-Day Sharpe Ratio', fontsize=13, fontweight='bold')
ax.set_xlabel('Date'); ax.set_ylabel('Sharpe'); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig('plot_rolling_sharpe.png', dpi=150); plt.show()


In [20]:
# Rolling 60-day Sharpe — selected best agents only
fig, ax = plt.subplots(figsize=(16, 5))

ret_df = result_best.pct_change().dropna()

for col in ret_df.columns:
    rs = ret_df[col].rolling(60).apply(
        lambda x: np.sqrt(252) * x.mean() / x.std() if x.std() > 0 else 0
    )
    ax.plot(
        rs.index,
        rs,
        label=col,
        lw=2.5 if col == "Nasdaq-100" else 1.2,
        ls="--" if col == "Nasdaq-100" else "-"
    )

ax.axhline(0, color="black", lw=0.8, ls=":")
ax.set_title("Rolling 60-Day Sharpe Ratio — Main Comparison Strategies", fontsize=13, fontweight="bold")
ax.set_xlabel("Date")
ax.set_ylabel("Sharpe")
ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig("plot_rolling_sharpe_selected_best.png", dpi=150)
plt.show()

In [21]:
# Drawdown
fig, ax = plt.subplots(figsize=(16, 5))
for col in result.columns:
    cum = (1 + result[col].pct_change().dropna()).cumprod()
    dd  = (cum - cum.cummax()) / cum.cummax() * 100
    ax.plot(dd.index, dd, label=col,
            lw=2.5 if col=='Nasdaq-100' else 1.2,
            ls='--' if col=='Nasdaq-100' else '-')
ax.set_title('Drawdown Over Time (%)', fontsize=13, fontweight='bold')
ax.set_xlabel('Date'); ax.set_ylabel('Drawdown (%)'); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig('plot_drawdown.png', dpi=150); plt.show()


In [21]:
# Drawdown — selected best agents only
fig, ax = plt.subplots(figsize=(16, 5))

for col in result_best.columns:
    cum = (1 + result_best[col].pct_change().dropna()).cumprod()
    dd = (cum - cum.cummax()) / cum.cummax() * 100

    ax.plot(
        dd.index,
        dd,
        label=col,
        lw=2.5 if col == "Nasdaq-100" else 1.2,
        ls="--" if col == "Nasdaq-100" else "-"
    )

ax.set_title("Drawdown Over Time (%) — Main Comparison Strategies", fontsize=13, fontweight="bold")
ax.set_xlabel("Date")
ax.set_ylabel("Drawdown (%)")
ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig("plot_drawdown_selected_best.png", dpi=150)
plt.show()

## Part 13 · Hyperparameter Sweep (Optional)

Tests PPO-DeepSeek 10% across turbulence/hmax/cost configs.
Re-run this cell independently — doesn't affect Parts 6–12.


In [ ]:
sweep_configs = [
    {'turb': 50,  'hmax': 100, 'cost': 0.001},
    {'turb': 70,  'hmax': 100, 'cost': 0.001},  # default
    {'turb': 100, 'hmax': 100, 'cost': 0.001},
    {'turb': 70,  'hmax': 50,  'cost': 0.001},
    {'turb': 70,  'hmax': 20,  'cost': 0.001},
    {'turb': 70,  'hmax': 100, 'cost': 0.002},
]

sweep_rows = []
for cfg in sweep_configs:
    label = f"turb={cfg['turb']} hmax={cfg['hmax']} cost={cfg['cost']}"
    print(f"  {label}")
    sd = len(trade_sentiment.tic.unique())
    ss = 1 + 2*sd + (len(INDICATORS)+1)*sd
    kw = dict(hmax=cfg['hmax'], initial_amount=1_000_000,
              num_stock_shares=[0]*sd, buy_cost_pct=[cfg['cost']]*sd,
              sell_cost_pct=[cfg['cost']]*sd, state_space=ss, stock_dim=sd,
              tech_indicator_list=INDICATORS, action_space=sd, reward_scaling=REWARD_SCALING)
    env = StockTradingEnv_llm(df=trade_sentiment, turbulence_threshold=cfg['turb'],
                               risk_indicator_col='vix', **kw)
    if ppo_llm_10 is None:
        print("    skipped — model not loaded"); continue
    vals, *_ = drl_predict(ppo_llm_10, env)
    ret  = (vals[-1]/vals[0]-1)*100
    rets = pd.Series(vals).pct_change().dropna()
    sh   = np.sqrt(252)*rets.mean()/rets.std()
    cum  = (1+rets).cumprod()
    mdd  = ((cum-cum.cummax())/cum.cummax()).min()*100
    sweep_rows.append({'Config':label,'Return %':round(ret,2),'Sharpe':round(sh,3),'MaxDD %':round(mdd,2)})

if sweep_rows:
    sweep_df = pd.DataFrame(sweep_rows).set_index('Config')
    display(sweep_df.style
        .background_gradient(subset=['Return %','Sharpe'], cmap='RdYlGn')
        .background_gradient(subset=['MaxDD %'], cmap='RdYlGn_r')
        .set_caption('PPO-DeepSeek 10% sensitivity'))


  turb=50 hmax=100 cost=0.001


RuntimeError: mat1 and mat2 shapes cannot be multiplied (1x925 and 1009x512)

## Part 14 · Export

In [16]:
result.to_csv('portfolio_values_all.csv')
result_ppo.to_csv('portfolio_values_ppo.csv')
result_cppo.to_csv('portfolio_values_cppo.csv')
metrics.to_csv('metrics_all.csv')
metrics_ppo.to_csv('metrics_ppo.csv')
metrics_cppo.to_csv('metrics_cppo.csv')
print("All files saved ✓")


All files saved ✓
